# Week 1 · Day 1 — Website Summarizer

A first look at calling an LLM from Python. We'll:

1. Set up the OpenAI client and load our API key.
2. Send a simple chat completion ("tell me a joke").
3. Scrape a real web page down to clean text.
4. Ask the model to summarize that page in markdown.

> **Prerequisite:** copy `.env.example` to `.env` and add your `OPENAI_API_KEY`.

---

## 1. Setup & imports

In [ ]:
import os

from dotenv import load_dotenv  # loads variables from .env into os.environ
from IPython.display import Markdown, display  # render LLM output as markdown
from openai import OpenAI  # official OpenAI Python client

# local helpers, see scraper.py
from scraper import clean_html, fetch_website_content

## 2. Load API credentials

`load_dotenv` reads the key/value pairs from our local `.env` file into
environment variables. We fail fast if the OpenAI key is missing.

In [2]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

## 3. Your first chat completion

The Chat Completions API takes a list of `messages`. Each message has a
`role` (`system`, `user`, or `assistant`) and `content`. Here we send a
single user message and print the model's reply.

In [5]:
message = "Hi GPT it's my first time using you, can you tell me a joke?"
messages = [
    {"role": "user", "content": message},
]

In [ ]:
# Instantiate the client. It automatically picks up OPENAI_API_KEY from the
# environment, so no key needs to be passed explicitly.
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# The reply text lives at choices[0].message.content.
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))

## 4. Scrape a website

Before we can summarize a page, we need its content as plain text.
`fetch_website_content` (defined in [scraper.py](scraper.py)) downloads the
page, strips out scripts/styles/images, and returns the title plus visible
text.

In [ ]:
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
print(my_website_content)

Title: Azam Mustufa — Software Engineer

Azam Mustufa — Software Engineer
AZAM.DEV
Work
Experience
Connect
mail
System Operational · IBM, Bangalore
Software Engineer
at IBM
Software Engineer at IBM working on SAP SuccessFactors cloud integrations. I build backend systems with Java, Spring Boot, and Node.js, and focus on quality engineering, test automation, and distributed systems.
281+
LeetCode · Top 13%
5
Projects deployed
3×
Hackathon finalist
View Work
arrow_downward
Connect
Projects
Backend Systems
terminal
Live
Database Backup CLI Infrastructure
Production-grade automation CLI for async database backups — reduces manual deployment and migration time by 80%, handles 50GB migrations under 150MB memory with 99.9% success rate.
TypeScript
MongoDB
MySQL
Node.js
Linux
Bash
GitHub
arrow_outward
hub
Live
Project Management Engine
Scalable REST backend with 25+ endpoints and granular RBAC. API responses optimised to under 120ms via Redis caching. BullMQ async pipelines cut cache latency b

## 5. Summarize the page with an LLM

This is a classic **system + user prompt** pattern:

- The **system prompt** sets the model's role and output format.
- The **user prompt** carries the actual data (the scraped page text).

In [ ]:
# Defines the assistant's role and the format we want back. No interpolation
# needed here, so it's a plain string (not an f-string).
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)

In [ ]:
# The user prompt carries the scraped page text for the model to summarize.
user_prompt = f"Here is the content of the website: {my_website_content}"

In [ ]:
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    return response.choices[0].message.content

In [18]:
# Combine both prompts into the messages list the API expects.
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
# Send the system + user prompts together and render the markdown summary.
response = summarize_website_content(messages)

display(Markdown(data=f"**GPT:** {response}"))

**GPT:** # Summary of the Website

- Azam Mustufa is a Software Engineer based in Bangalore, currently working at IBM on SAP SuccessFactors cloud integrations.
- Focus areas: backend systems, Java and Spring Boot, Node.js, quality engineering, test automation, and distributed systems.
- Notable metrics: 30+ defects resolved during cloud integration testing; 15% improvement in release stability; 3× Hackathon finalist; 5 projects deployed; LeetCode with 281+ problems (Top 13%).
- Key projects showcased (backend and infrastructure): Production-grade Database Backup CLI, Project Management Engine, Distributed Image Processing Backend, HTTP Caching Proxy, and UPI Offline Mesh. Each project highlights scalable architectures and specific tech stacks.
- Technical stack includes: Java, JavaScript/TypeScript, Python, C++, SQL; Spring Boot, Spring Security, Node.js, Express, Bun, BullMQ, Mongoose; MongoDB, MySQL, Redis, RabbitMQ; Docker, Linux, CI/CD; testing with JUnit, Mockito, Selenium, Playwright, Cypress.
- Education: B.E. in Computer Science & Engineering from Visvesvaraya Technological University (CGPA 7.28/10); notable coursework in OS, networks, data structures, databases, software engineering; 3× Hackathon finalist.
- Availability: Accepting new requests; located in Bangalore, India; open to remote work.
- Contact and online presence: Email, resume, GitHub, LinkedIn, LeetCode profile.
- Site note: AZAM.DEV built with Nuxt 3.

## 6. Reuse the helper on another page

Now that `summarize_website_content` is a function, summarizing a different
site is just: scrape → build messages → call the helper.

In [25]:
content = fetch_website_content(url="https://cnn.com/")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Here is the content of the website: {content}"},
]

In [26]:
summary = summarize_website_content(messages)
display(Markdown(data=f"**GPT:** {summary}"))

**GPT:** - What the site is
  - CNN’s homepage for breaking news, latest news, and videos, with live updates and a mix of articles, analyses, and exclusive reports.

- Editions and regions
  - Editions include US, International, Arabic, and Español. Content is tailored to different regions.

- Main news categories covered
  - US news, World news, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather
  - Special topic areas: World Cup 2026, Ukraine-Russia War, Israel-Hamas War

- Content formats and features
  - Live TV and “Watch/Listen” options
  - Video content (CNN-exclusive videos, live updates, analyses)
  - Articles, investigations, and photo features
  - Special segments: CNN Exclusive, Live Updates, Analysis, Features, CNN Heroes, Unearthed, CNN 10, Podcasts, and games/quizzes

- Engagement and personalization
  - User accounts: sign in, settings, following topics, newsletters
  - Ad feedback prompts and feedback options
  - Follow CNN sections and customize content topics

- Multimedia and additional tools
  - CNN Fast, shows (A-Z), and various multimedia offerings
  - Downloadable app options (QR codes provided) for mobile access

- Notable coverage themes visible on the homepage
  - Global conflict and security (Ukraine-Russia, Israel-Hamas)
  - US politics and elections (Elections 2026, GOP/White House topics)
  - Economic and tech news, health developments, science discoveries
  - Climate, weather, and environmental stories
  - Sports and major events (e.g., World Cup coverage)

In short, the site aims to provide comprehensive, multimedia coverage of global and US news across many categories, with live updates, exclusive reporting, and personalized options via accounts and newsletters.

## 7. Selenium-based web scraping

The `requests`-based scraper only sees the initial HTML. For sites that
render their content with JavaScript (single-page apps, infinite scroll,
etc.), we need a real browser. **Selenium** drives a headless Chrome instance
so we capture the fully-rendered page.

The `WebsiteSummarizer` class below bundles the whole flow — fetch, summarize,
display — behind a small object.

> **Requires:** `selenium` (already in `requirements.txt`). Selenium 4.6+ auto-downloads
> a matching chromedriver, so no manual driver setup is needed — just a local Chrome install.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class WebsiteSummarizer:
    """Fetch a JS-rendered page with Selenium and summarize it with an LLM.

    Typical usage:
        ws = WebsiteSummarizer(url)
        ws.selenium_fetch_content()
        ws.summarize_content()
        ws.display_summary()
    """

    def __init__(self, url: str):
        self.url = url
        self.content: str | None = None 
        self.summary: str | None = None 
        self.openai = OpenAI()

    def selenium_fetch_content(self) -> None:
        """Load the page in headless Chrome and capture the cleaned text."""
        options = Options()
        options.add_argument("--headless=new")  # run without opening a window
        options.add_argument(
            "User-Agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        driver = webdriver.Chrome(options=options)
        try:
            driver.get(url=self.url)
            driver.implicitly_wait(5)
            # Strip the rendered HTML down to title + visible text (same cleaning
            # as scraper.py) so we send far fewer tokens to the model.
            self.content = clean_html(driver.page_source)
        except Exception as e:
            print(f"Error fetching content with Selenium: {e}")
        finally:
            driver.quit()

    def summarize_content(self) -> None:
        """Summarize the fetched content using the LLM."""
        if not self.content:
            raise ValueError(
                "Content not fetched yet. Call selenium_fetch_content() first."
            )

        system_prompt = (
            "You are a helpful assistant that summarizes website content. "
            "Provide a concise summary of the main points and topics covered on the "
            "website. Respond in markdown format."
        )
        user_prompt = f"Here is the content of the website: {self.content}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        response = self.openai.chat.completions.create(
            model="gpt-5-nano",
            messages=messages,  # type: ignore
        )
        self.summary = response.choices[0].message.content

    def display_summary(self) -> None:
        """Render the summary as markdown."""
        if not self.summary:
            raise ValueError(
                "Summary not generated yet. Call summarize_content() first."
            )
        display(Markdown(data=self.summary))

In [20]:
ws = WebsiteSummarizer(url="https://bseindia.com/")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

Here’s a concise summary of the main points and topics covered on the site:

- What the site is
  - Official BSE (Bombay Stock Exchange) site providing live stock market updates, Sensex indices, real-time stock prices, and market data.
  - Coverage extends to stock/indices, derivatives (equity, currency, commodity, interest rate), currencies, and commodities derivatives.
  - Includes company news, results, and corporate actions.

- Core market data sections
  - Live market updates: BSE Sensex, index levels, and real-time stock prices.
  - Market Derivatives: Market Summary, Market Watch, and various derivative pages (Equity, Currency, IRD, Commodity, Debt; Top Turnover; Block Deals; Listings).
  - Indices and performance: Indices Watch, Gainers/Losers, 52-week highs/lows, turnover, and open interest data.
  - Market Turnover and trading activity: Highlights of contracts traded, notional values, and trades.

- Market statistics and analytics
  - Market Statistics: broad metrics such as market capitalization, number of stocks at 52-week highs/lows, advances/declines, and related charts.
  - Listing Statistics: overview of listing-related data.
  - Bhav Copy access and historical/data-driven insights (through various stat pages and charts).

- Investor resources and notices
  - Notices and Circulars, Media Releases, Trading Holidays.
  - Corporate announcements, board meetings, and corporate actions.
  - Public issues and IPO-related pages; MF/ETF market data and settlement/calendar information.

- Tools and resources for investors
  - Get Quote, historical data access, charting tools, and market data products.
  - IRD (Interest Rate Derivatives) tools, options calculators, and other derivatives-related utilities.
  - Payments to BSE, BSEPlus, SME platforms, startups, and related services.

- Navigation, language, and access options
  - Multi-language/group-site navigation (English with options to switch languages and links to Hindi, Marathi, Gujarati).
  - “Old Website” links for quick access to legacy interfaces.
  - Quick links in the footer for rapid access to key resources (auditors, notices, policies, sitemap, social channels).

- Footer and additional links
  - Quick Links include auditor empanelment, master circulars, regulatory resources, and access to charting, payments, and contact options.
  - Social media and regulatory-facing links (Facebook, Twitter, LinkedIn, YouTube, RSS).

- User interface notes
  - Search functionality for securities (smart search) and a prominent “FeedBack/Feedback” option.
  - Banner sections and content areas for news sliders, market stats, and trend widgets.
  - The page uses extensive CSS for layout and responsive design, along with an Angular-based structure (evident from tag names like app-*, ng- attributes).

In short: the site is a comprehensive hub for live BSE market data, derivatives, key statistics, and investor information, complemented by tools, notices, and access to both current and legacy interfaces.